# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: 
Date: 

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
from pathlib import Path

ROOT = Path.cwd()
CHECKS = [(ROOT / ".env", "local configuration"), (ROOT / ".env.example", "safe template")]
print(f"Homework root: {ROOT}")
for path, note in CHECKS:
    print(f"[{'OK' if path.exists() else 'MISS'}] {path.name:<14} {note}")
assert all(path.exists() for path, _ in CHECKS)

Homework root: /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework04
[OK] .env           local configuration
[OK] .env.example   safe template


In [3]:
import datetime as dt
import os

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
RAW = ROOT / os.getenv("DATA_DIR_RAW", "data/raw")
RAW.mkdir(parents=True, exist_ok=True)
RUN_STAMP = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
print("Raw directory:", RAW)
print("Run stamp:", RUN_STAMP)

Raw directory: /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework04/data/raw
Run stamp: 20260817-103817


## Helpers (use or modify)

In [4]:
def save_csv(frame: pd.DataFrame, prefix: str, **meta):
    middle = "_".join(f"{key}-{value}" for key, value in meta.items())
    path = RAW / f"{prefix}_{middle}_{RUN_STAMP}.csv"
    frame.to_csv(path, index=False)
    print("Saved", path)
    return path


def validate(frame: pd.DataFrame, required, min_rows=1):
    missing = [column for column in required if column not in frame.columns]
    result = {
        "missing": missing,
        "shape": frame.shape,
        "na_by_column": frame.isna().sum().to_dict(),
        "duplicate_rows": int(frame.duplicated().sum()),
    }
    assert not missing, f"Missing columns: {missing}"
    assert len(frame) >= min_rows, f"Expected at least {min_rows} rows"
    return result

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [5]:
SYMBOL = "SPY"
raw_api = yf.download(
    SYMBOL,
    start="2026-01-01",
    auto_adjust=True,
    progress=False,
    actions=False,
    threads=False,
)
assert not raw_api.empty, "yfinance returned no data"
if isinstance(raw_api.columns, pd.MultiIndex):
    raw_api.columns = raw_api.columns.get_level_values(0)
df_api = raw_api.reset_index().rename(columns=str.lower)
df_api = df_api[["date", "open", "high", "low", "close", "volume"]]
df_api["date"] = pd.to_datetime(df_api["date"], errors="raise")
for column in ["open", "high", "low", "close", "volume"]:
    df_api[column] = pd.to_numeric(df_api[column], errors="raise")
v_api = validate(df_api, ["date", "open", "high", "low", "close", "volume"], min_rows=20)
assert df_api["date"].is_monotonic_increasing and df_api["date"].is_unique
assert (df_api[["open", "high", "low", "close"]] > 0).all().all()
print(v_api)
df_api.head()

{'missing': [], 'shape': (156, 6), 'na_by_column': {'date': 0, 'open': 0, 'high': 0, 'low': 0, 'close': 0, 'volume': 0}, 'duplicate_rows': 0}


Price,date,open,high,low,close,volume
0,2026-01-02,682.085206,683.239047,676.226327,679.558594,89377200
1,2026-01-05,682.910779,685.785516,682.751651,684.084534,71927200
2,2026-01-06,684.293460,688.660268,684.144289,688.152954,69273800
3,2026-01-07,688.530942,690.291605,685.676118,685.934753,75588300
4,2026-01-08,685.178757,686.969230,683.855771,685.865112,64019200


In [6]:
api_path = save_csv(df_api, prefix="api", source="yfinance", symbol=SYMBOL)

Saved /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework04/data/raw/api_source-yfinance_symbol-SPY_20260817-103817.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [7]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "FRE5040-course-project/1.0"}
response = requests.get(SCRAPE_URL, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
table = soup.select_one("table#constituents")
assert table is not None, "S&P 500 constituents table not found"
rows = table.select("tr")
header = [cell.get_text(" ", strip=True) for cell in rows[0].find_all(["th", "td"])]
data = [
    [cell.get_text(" ", strip=True) for cell in row.find_all(["th", "td"])]
    for row in rows[1:]
]
data = [row for row in data if len(row) == len(header)]
df_scrape = pd.DataFrame(data, columns=header)
df_scrape = df_scrape[["Symbol", "Security", "GICS Sector", "GICS Sub-Industry", "CIK"]]
df_scrape["CIK"] = pd.to_numeric(df_scrape["CIK"], errors="raise")
v_scrape = validate(df_scrape, ["Symbol", "Security", "GICS Sector", "GICS Sub-Industry", "CIK"], min_rows=500)
assert df_scrape["Symbol"].str.len().gt(0).all()
assert pd.api.types.is_numeric_dtype(df_scrape["CIK"])
print(v_scrape)
df_scrape.head()

{'missing': [], 'shape': (503, 5), 'na_by_column': {'Symbol': 0, 'Security': 0, 'GICS Sector': 0, 'GICS Sub-Industry': 0, 'CIK': 0}, 'duplicate_rows': 0}


,Symbol,Security,GICS Sector,GICS Sub-Industry,CIK
0,MMM,3M,Industrials,Industrial Conglomerates,66740
1,AOS,A. O. Smith,Industrials,Building Products,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1800
3,ABBV,AbbVie,Health Care,Biotechnology,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,1467373


In [8]:
scrape_path = save_csv(df_scrape, prefix="scrape", site="wikipedia", table="sp500-constituents")

Saved /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework04/data/raw/scrape_site-wikipedia_table-sp500-constituents_20260817-103817.csv


## Documentation

- **API source:** Yahoo Finance through `yfinance.download`; SPY; start `2026-01-01`; daily adjusted OHLCV; open end date.
- **Scrape source:** Wikipedia's public “List of S&P 500 companies” page; HTML table `#constituents`; requested with an identifying course user agent.
- **Validation:** HTTP success, table presence, required columns, minimum rows, parsed date/numeric types, null counts, duplicate counts, ordered unique API dates, positive prices, nonempty symbols, and numeric CIK.
- **Saved outputs:** timestamped raw CSV files under `data/raw/`.
- **Assumptions and risks:** Yahoo and Wikipedia may change availability or schema; adjusted prices can be revised; HTML selectors are fragile; these sources are appropriate for a course exercise but not contractual institutional feeds.
- **Secrets:** neither chosen source requires a key. Local `.env` is ignored and only `.env.example` is committed.

In [9]:
assert api_path.is_file() and scrape_path.is_file()
assert pd.read_csv(api_path).shape == df_api.shape
assert pd.read_csv(scrape_path).shape == df_scrape.shape
print('All Stage 04 checks passed.')

All Stage 04 checks passed.


## AI Assistance

AI assistance was used to implement and verify the ingestion workflow. Hanson Sun reviewed the sources, assumptions, and outputs.